In [9]:
# Cellule 1 : Imports
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from pathlib import Path
import sys

# Chemins compatibles Windows/WSL
project_root = Path(r'C:\Users\nidha\Desktop\xai_clinical_prediction')
sys.path.append(str(project_root / 'src'))

# Cellule 2 : Chargement
PROCESSED_DIR = project_root / 'data' / 'processed'

clinical_df = pd.read_csv(PROCESSED_DIR / 'clinical_explored.csv')
mrna_df     = pd.read_csv(PROCESSED_DIR / 'mrna_explored.csv', index_col='SAMPLE_ID')
print(clinical_df.columns.tolist())
# Cellule 3 — inchangée
def create_target(clinical_df):
    df = clinical_df.copy()
    df['OS_MONTHS'] = pd.to_numeric(df['OS_MONTHS'], errors='coerce')
    df['DECEASED'] = df['OS_STATUS'].map({'0:LIVING': 0, '1:DECEASED': 1})
    stage_col = 'AJCC_PATHOLOGIC_TUMOR_STAGE'
    df['COMPLICATION'] = (
        ((df['DECEASED'] == 1) & (df['OS_MONTHS'] < 24)) |
        df[stage_col].astype(str).str.startswith(('Stage III', 'Stage IV'))
    ).astype(int)
    return df['COMPLICATION']

y = create_target(clinical_df)
print(f"Distribution des complications:\n{y.value_counts()}")

# Cellule 4 : Préparation des features cliniques — AGE nettoyé
clinical_features = [
    'AGE',
    'AJCC_PATHOLOGIC_TUMOR_STAGE',
    'HISTOLOGICAL_DIAGNOSIS',
    'ER_STATUS_BY_IHC',
    'PR_STATUS_BY_IHC',
    'IHC_HER2',
]
X_clinical = clinical_df[clinical_features].copy()

# Nettoyage AGE — remplacer les valeurs non numériques par la médiane
X_clinical['AGE'] = pd.to_numeric(X_clinical['AGE'], errors='coerce')
age_median = X_clinical['AGE'].median()
X_clinical['AGE'] = X_clinical['AGE'].fillna(age_median)

# Encodage stade tumoral
le_stage = LabelEncoder()
X_clinical['TUMOR_STAGE_ENC'] = le_stage.fit_transform(
    X_clinical['AJCC_PATHOLOGIC_TUMOR_STAGE'].fillna('Unknown')
)

# Encodage grade histologique
le_grade = LabelEncoder()
X_clinical['GRADE_ENC'] = le_grade.fit_transform(
    X_clinical['HISTOLOGICAL_DIAGNOSIS'].fillna('Unknown')
)

# Encodage statuts IHC
status_map = {
    'Positive': 1, 'Negative': 0, 'Indeterminate': 0.5,
    'Equivocal': 0.5, '[Not Evaluated]': np.nan,
    '[Not Available]': np.nan, '[Unknown]': np.nan,
}
for col in ['ER_STATUS_BY_IHC', 'PR_STATUS_BY_IHC', 'IHC_HER2']:
    X_clinical[col] = X_clinical[col].map(status_map).fillna(0)

X_clinical = X_clinical[[
    'AGE', 'TUMOR_STAGE_ENC', 'GRADE_ENC',
    'ER_STATUS_BY_IHC', 'PR_STATUS_BY_IHC', 'IHC_HER2'
]]

# Vérification : aucune valeur non numérique ne doit subsister
print(X_clinical.dtypes)
print(f"\nValeurs manquantes : {X_clinical.isnull().sum().sum()}")
print(f"✅ Features cliniques : {X_clinical.shape}")

# Cellule 5 : Préparation des features génomiques
# Sélection des gènes les plus variés
gene_vars = mrna_df.var().sort_values(ascending=False)
top_genes = gene_vars.head(500).index  # Top 500 gènes les plus variés
X_genomic = mrna_df[top_genes]

# Normalisation log2 pour les données RNA-Seq
X_genomic = np.log2(X_genomic + 1)

# Cellule 6 : Alignement des échantillons — version finale

# 1. Dédoublonner mrna_df (garder un seul échantillon par patient)
mrna_df_dedup = mrna_df.copy()
mrna_df_dedup.index = [s[:12] for s in mrna_df_dedup.index]  # Tronquer à 12 chars
mrna_df_dedup = mrna_df_dedup[~mrna_df_dedup.index.duplicated(keep='first')]

print(f"mrna_df original   : {mrna_df.shape[0]}")
print(f"mrna_df dédoublonné: {mrna_df_dedup.shape[0]}")

# 2. IDs communs
clinical_ids = clinical_df['PATIENT_ID'].values
common_ids   = sorted(set(clinical_ids) & set(mrna_df_dedup.index))
print(f"Échantillons communs: {len(common_ids)}")

# 3. Masque sur clinical (basé sur l'ordre original de clinical_df)
mask_clinical = clinical_df['PATIENT_ID'].isin(common_ids)

# 4. Aligner X_clinical et y (déjà construits) sur le masque
X_clinical = X_clinical[mask_clinical].reset_index(drop=True)
y          = y[mask_clinical].reset_index(drop=True)

# 5. Aligner mrna dans le même ordre que clinical_df filtré
ordered_ids = clinical_df[mask_clinical]['PATIENT_ID'].values
mrna_aligned = mrna_df_dedup.loc[ordered_ids]

# 6. Reconstruire X_genomic depuis mrna aligné
gene_vars = mrna_aligned.var().sort_values(ascending=False)
top_genes = gene_vars.head(500).index
X_genomic = np.log2(mrna_aligned[top_genes] + 1).reset_index(drop=True)

print(f"\nX_clinical : {X_clinical.shape}")
print(f"X_genomic  : {X_genomic.shape}")
print(f"y          : {y.shape}")
assert X_clinical.shape[0] == X_genomic.shape[0] == y.shape[0], "❌ Tailles incohérentes !"
print("✅ Alignement OK")

# Cellule 7 : Feature selection génomique
selector = SelectKBest(f_classif, k=100)  # Top 100 gènes les plus corrélés à la cible
X_genomic_selected = selector.fit_transform(X_genomic, y)
selected_genes = X_genomic.columns[selector.get_support()]
print(f"Gènes sélectionnés: {len(selected_genes)}")

# Cellule 8 : Fusion des features
X_combined = pd.concat([
    X_clinical.reset_index(drop=True),
    pd.DataFrame(X_genomic_selected, columns=selected_genes)
], axis=1)

print(f"Features combinées: {X_combined.shape}")

# Cellule 9 : Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

# Cellule 10 : Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Cellule 11 : Sauvegarde — chemins corrigés
np.save(PROCESSED_DIR / 'X_train.npy', X_train_scaled)
np.save(PROCESSED_DIR / 'X_test.npy',  X_test_scaled)
np.save(PROCESSED_DIR / 'y_train.npy', y_train)
np.save(PROCESSED_DIR / 'y_test.npy',  y_test)

pd.DataFrame(X_train_scaled, columns=X_combined.columns).to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
pd.DataFrame(X_test_scaled,  columns=X_combined.columns).to_csv(PROCESSED_DIR / 'X_test.csv',  index=False)

print("✅ Données prétraitées sauvegardées dans :", PROCESSED_DIR)

['OTHER_PATIENT_ID', 'PATIENT_ID', 'FORM_COMPLETION_DATE', 'PROSPECTIVE_COLLECTION', 'RETROSPECTIVE_COLLECTION', 'SEX', 'MENOPAUSE_STATUS', 'RACE', 'ETHNICITY', 'HISTORY_OTHER_MALIGNANCY', 'HISTORY_NEOADJUVANT_TRTYN', 'TUMOR_STATUS', 'RADIATION_TREATMENT_ADJUVANT', 'PHARMACEUTICAL_TX_ADJUVANT', 'HISTOLOGICAL_SUBTYPE', 'INITIAL_PATHOLOGIC_DX_YEAR', 'AGE', 'METHOD_OF_INITIAL_SAMPLE_PROCUREMENT', 'METHOD_OF_INITIAL_SAMPLE_PROCUREMENT_OTHER', 'SURGICAL_PROCEDURE_FIRST', 'FIRST_SURGICAL_PROCEDURE_OTHER', 'PATH_MARGIN', 'SURGERY_FOR_POSITIVE_MARGINS', 'SURGERY_FOR_POSITIVE_MARGINS_OTHER', 'MARGIN_STATUS_REEXCISION', 'STAGING_SYSTEM', 'STAGING_SYSTEM_OTHER', 'MICROMET_DETECTION_BY_IHC', 'LYMPH_NODES_EXAMINED', 'LYMPH_NODE_EXAMINED_COUNT', 'LYMPH_NODES_EXAMINED_HE_COUNT', 'LYMPH_NODES_EXAMINED_IHC_COUNT', 'AJCC_STAGING_EDITION', 'AJCC_TUMOR_PATHOLOGIC_PT', 'AJCC_NODES_PATHOLOGIC_PN', 'AJCC_METASTASIS_PATHOLOGIC_PM', 'AJCC_PATHOLOGIC_TUMOR_STAGE', 'METASTATIC_SITE_PATIENT', 'METASTATIC_SITE_OTH